# Constrained dispatch forecasting — hist (2021–2026), Colab GPU

Runtime > Change runtime type > **GPU** (A100/L4 if offered). The repo ships the
prepared training data (`data/preprocessed/hist/...`, 17 MB) and the mined stress
episodes, so cloning is all the setup there is. Every training run checkpoints per
epoch and auto-resumes — a disconnect costs at most one epoch; just re-run the cell.

## The problem

Predict the 5-min VIC dispatch vector **P** = (hydro, coal, gas_steam, gas_ocgt,
batt_chg, batt_dis) one step ahead, such that every prediction **satisfies the grid
physics by construction**:

- **balance** — predicted net demand equals the signed dispatch sum:
  `SIGN·P − D_t = 0` with `SIGN = (1,1,1,1,−1,1)` (battery charging is a load)
- **ramps** — per-generator asymmetric limits `−R_dn ≤ P_i,t − P_i,t−1 ≤ R_up`
  (data-derived, `ml/check_caps.py::RAMPS`)
- **floors** — `P_i ≥ 0` (without this, idle gas_steam with ramp-down 500 could
  legally emit −500 MW)

Output vector everywhere below: **y = (D_t, P_1..P_6)** — 7 numbers, demand first.
Two hard-constraint mechanisms are trained and compared against their bare
backbones and against 5-min persistence (hist test bar: **WAPE 0.1084**).

## Approach 1 — RAYEN hard layer (`--arch *_rayen`), trained through

Konstantinov/RAYEN-style feasible-set parameterisation (`rayen-hardconvex.pdf`,
`lib/models.py::RayenHead`) replacing the network's final layer:

1. **anchor** `p = (SIGN·P_t−1, P_t−1)` — the previous step's actual dispatch. It
   satisfies everything at once: its sum equals its demand entry, zero change
   respects every ramp, and it is ε-interior to the floors (ε = 0.5 MW).
2. backbone emits a **raw direction** r ∈ R⁷ and a scalar s (8 raw outputs; the
   iTransformer keeps input RevIN but output denorm OFF — window means added onto
   direction components would corrupt them).
3. **stay on the balance plane**: r ← r − (a·r / a·a)·a with a = (−1, SIGN).
4. **respect walls**: α\* = distance along r to the nearest ramp/floor wall.
5. **output** y = p + σ(s)·α\*·r — feasible at any weights, exact gradients.

Zero direction = persistence, so the net learns bounded constrained *deviations*
from the strongest baseline. Trained with MSE on the 7-dim target
[nd_t (x-scaled), 6 dispatch (y-scaled)].

## Approach 2 — decision rules (`--arch *_task7` + post-hoc blend)

Constante Flores / Chen / Li 2025 (`hardlinearconstraints.pdf`,
`lib/decision_rule.py`). Two subnetworks, blended per prediction:

- **task network** — the same backbones with a plain 7-output head, trained for
  accuracy as usual (no constraint machinery in training). At inference its raw
  ŷ is snapped onto the balance plane by the closed-form orthogonal projection
  (paper eq. 16): ŷ ← ŷ − a(aᵀŷ)/(aᵀa).
- **safe network** — a fixed linear rule y_SN = F·x with x = [1, P_t−1, nd_t−1],
  found ONCE by the LP (paper eq. 14, per-row LP duality): maximise the
  worst-case slack t over the whole input box X subject to `aᵀF = 0` and every
  ramp/floor inequality holding for **all** x ∈ X. Solved by scipy/HiGHS in
  seconds; certificate t\* = **29.45 MW** (binding wall: gas_steam R_up/2), and
  the solution is interpretable: persistence + fixed positive offsets.
- **blend** — per sample, α = max over violated inequalities of
  −s_TN/(s_SN − s_TN) (paper eq. 4; α = 0 when the task output violates
  nothing). y = (1−α)·y_TN + α·y_SN is feasible by convexity; the equality is
  preserved because both endpoints satisfy it.

The training script fits the LP and reports the +DR row automatically after each
`*_task7` run; the bare task-net row doubles as the unconstrained Pareto baseline.

## Evaluation protocol

| check | where | why |
| --- | --- | --- |
| demand balance | regular test set, teacher-forced (Jan–Jul 2026, 53,568 windows) | balance is per-step; TF isolates the mechanism |
| ramp rate | **simulated test set**: closed-loop autoregressive rollout over the 15 test-split stress episodes (`constraints/stress_episodes.json` — record peaks, price spikes, battery-cycling days) | in deployment ramps bind on the model's *own* trajectory |

Metrics (`constraints/eval_hist_models.py`): WAPE (+ per-target), nd_WAPE (D_t
forecast quality), `n_demand` @own / @act (steps with |mismatch| > 10 MW),
`mismatch_act_pct` = 100·Σ|SIGN·pred − nd_act| / Σ|nd_act|, `n_ramp` (+0.6 MW
tol), `ramp_excess_pct` = 100·Σ(MW beyond limit) / Σ|pred| (raw violation MWh as
a share of dispatched MWh), `n_neg`, and battery feasibility = per-day (TF) /
per-episode (CL) SOC-window existence at η_rt = 0.834, cap 4,736 MWh.

In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""  # repo is PRIVATE: paste a fine-grained READ-ONLY token (Contents: read)
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("energy_modelling"):
    !git clone -q $url
%cd energy_modelling
!git pull -q
!nvidia-smi -L  # scipy/HiGHS for the safe-network LP ships with Colab

In [ ]:
# optional: mount Drive so weights/metrics survive the VM and sync back to you
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/energy_runs"
os.makedirs(OUT, exist_ok=True)

## 0 · Sanity: constraint layers are feasible at random weights

The guarantees are architectural — they must hold before a single gradient step.
~2 min; also certifies the safe-network LP (expect `t* = 29.45 MW`).

In [ ]:
!python constraints/test_constraint_layers.py

## 1 · Strided pilots (~10 min total)

Protocol (plan.md): trained-through layers have failed here before, so every arch
gets a cheap stride-12 pilot before a full run. Look for: rayen self-check shows
`ramp_viol=0, neg=0`, balance residual ~1e-3 MW; task7 +DR row likewise; and no
val-loss explosion. Pilot WAPEs are optimistic-noisy — do not compare them.

In [ ]:
for arch in ["lstm_rayen", "itransformer_rayen", "lstm_task7", "itransformer_task7"]:
    !python ml/train_hist.py --arch $arch --seed 0 --stride 12 --epochs 8 --out /content/pilots

## 2 · Full training runs

Recipes (apples-to-apples with the reference models): lstm\*: 40 epochs, patience
6, batch 128; itransformer\*: 60 / 10 / 64; lr 1e-4. On an A100 expect roughly
20–40 min per LSTM and 1–2 h per iTransformer. Each cell is independent —
re-running resumes from the last epoch checkpoint. `*_task7` cells print three
result blocks: bare task net, safe-LP certificate, and the +DR constrained row.

In [ ]:
# approach 1 — RAYEN hard layer, LSTM backbone
!python ml/train_hist.py --arch lstm_rayen --seed 0 --out $OUT

In [ ]:
# approach 1 — RAYEN hard layer, iTransformer (input RevIN, output denorm off)
!python ml/train_hist.py --arch itransformer_rayen --seed 0 --out $OUT

In [ ]:
# approach 2 — task net LSTM (bare row = Pareto baseline; +DR row = decision rules)
!python ml/train_hist.py --arch lstm_task7 --seed 0 --out $OUT

In [ ]:
# approach 2 — task net iTransformer (full RevIN: outputs are levels, not directions)
!python ml/train_hist.py --arch itransformer_task7 --seed 0 --out $OUT

## 3 · Evaluation — both tables

One command: persistence benchmark + every checkpoint found in `$OUT`
(rayen models, bare task nets, +DR rows rebuilt from the saved safe-F).

- **TF table** (demand balance, regular test set) → `constraints/results/hist_tf.md`
- **CL table** (ramp rate, simulated test set = closed-loop stress episodes)
  → `constraints/results/hist_cl_stress.md`

Reading guide: constrained rows must show `bal_own_max ≈ 0`, `n_ramp = 0`,
`n_neg = 0` — the interesting question is their WAPE vs the bare rows and the
0.1084 persistence bar. `alpha_frac_active` (in the JSONs) tells how often the DR
safe net actually intervened on a converged task net (expect ≈ 0).

In [ ]:
!python constraints/eval_hist_models.py --ckpt-dir $OUT
from IPython.display import Markdown, display
for t in ["hist_tf.md", "hist_cl_stress.md"]:
    display(Markdown(open(f"constraints/results/{t}").read()))

In [ ]:
# no Drive? zip and download instead
# !zip -r runs.zip $OUT constraints/results && python -c "from google.colab import files; files.download('runs.zip')" 